# 05 · Load the gold tables into MongoDB Atlas (+ Atlas Search)
**Why Atlas:** the gold tables are small, nested-friendly and searched by text (airport names, cascade descriptions), so they live in documents, while the 100M-row flight tables stay in Parquet.

| Collection | Documents | Source |
|---|---|---|
| `airports` | one per airport: name, city, location, cascade stats | OurAirports + `tables/cascades_by_airport.csv` |
| `summary_airport_hour` | one per airport and local hour | gold table from notebook 03 |
| `cascade_sample` | a random sample of cascades, each with a text description | gold table `cascades` |
| `by_year`, `by_hour`, `weather_split`, `delay_classes`, `bts_cause` | a few rows each | `tables/*.csv` from notebook 04 |

In [2]:
import os, re, json, subprocess, sys
from pathlib import Path
import pandas as pd, duckdb
try:
    import pymongo, bson
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "pymongo"], check=True)
    import pymongo, bson
from pymongo import MongoClient, InsertOne
from pymongo.operations import SearchIndexModel

PROJECT = Path(os.path.expanduser("~/Desktop/ADSP32030_Scalable/Project"))
PROP    = PROJECT/"data/propagation"
TABLES  = PROP/"tables"                                   # written by notebook 04
AIRPORTS_CSV = PROJECT/"data/weather/meta/airports.csv"   # OurAirports, downloaded earlier

SAMPLE_N = 50_000          # cascades to load as documents (random sample, fixed seed)
MAX_MB   = 120             # stop if the estimated size of everything below exceeds this (M0 limit is 512 MB for the whole cluster)

# read .env without extra packages: KEY=value per line
env = {}
if (PROJECT/".env").exists():
    for line in (PROJECT/".env").read_text().splitlines():
        if line.strip() and not line.lstrip().startswith("#") and "=" in line:
            k, v = line.split("=", 1); env[k.strip()] = v.strip()
URI, DBNAME = env.get("ATLAS_URI"), env.get("ATLAS_DB", "flightdelay")
DRY_RUN = not URI
print("DRY RUN (no ATLAS_URI found): documents are built but nothing is sent" if DRY_RUN else f"connecting to database '{DBNAME}'")

connecting to database 'flightdelay'


## 1 · Build the documents

In [3]:
con = duckdb.connect()
q = lambda sql: con.execute(sql).df()
need = ["cascades_by_airport", "cascades_by_year", "cascades_by_hour", "weather_split", "delay_classes", "bts_cause_share"]
missing = [n for n in need if not (TABLES/f"{n}.csv").exists()]
assert not missing, f"run notebook 04 first (missing {missing} in {TABLES})"

# airports: OurAirports names/locations + our cascade statistics
oa = pd.read_csv(AIRPORTS_CSV, low_memory=False)
oa = (oa[oa.iata_code.notna()].sort_values("type", key=lambda s: s.map({"large_airport":0,"medium_airport":1}).fillna(2))
        .drop_duplicates("iata_code")[["iata_code","name","municipality","iso_region","latitude_deg","longitude_deg"]])
stats = pd.read_csv(TABLES/"cascades_by_airport.csv")
ap = stats.merge(oa, left_on="airport", right_on="iata_code", how="left")
airports = [{"_id": r.airport, "iata": r.airport, "name": None if pd.isna(r["name"]) else r["name"],
             "city": None if pd.isna(r.municipality) else r.municipality,
             "region": None if pd.isna(r.iso_region) else r.iso_region,
             "location": None if pd.isna(r.latitude_deg) else {"type": "Point", "coordinates": [float(r.longitude_deg), float(r.latitude_deg)]},
             "cascades": int(r.cascades), "departures": int(r.departures), "cascades_per_1000_departures": round(float(r.per_1000), 2)}
            for _, r in ap.iterrows()]
for d in airports: d["name"] = d["name"] or d["iata"]
for d in airports:
    if d["location"] is None: del d["location"]
    if d["city"] is None: del d["city"]
    if d["region"] is None: del d["region"]

# hour summary (gold table from notebook 03)
summ = q(f"SELECT * FROM read_parquet('{PROP/'summary_airport_hour'}/*.parquet')")
summary_docs = summ.astype({"local_hour": int, "cascades": int}).to_dict("records")

# small tables
small = {"by_year": pd.read_csv(TABLES/"cascades_by_year.csv"), "by_hour": pd.read_csv(TABLES/"cascades_by_hour.csv"),
         "weather_split": pd.read_csv(TABLES/"weather_split.csv"), "delay_classes": pd.read_csv(TABLES/"delay_classes.csv"),
         "bts_cause": pd.read_csv(TABLES/"bts_cause_share.csv").rename(columns={"Unnamed: 0": "cause"})}
small_docs = {k: json.loads(v.to_json(orient="records")) for k, v in small.items()}

# random sample of cascades, each with a text description for Atlas Search
name_of = {a["iata"]: a["name"] for a in airports}
city_of = {a["iata"]: a.get("city", "") for a in airports}
cs = q(f"""SELECT Tail_Number, chain_id, root_origin, root_dep_delay, root_sched_dep_utc, chain_len, downstream_inherited_min, Year, root_local_hour
           FROM read_parquet('{PROP/'cascades'}/*.parquet') USING SAMPLE {SAMPLE_N} ROWS (reservoir, 42)""")
def describe(r):
    return (f"{name_of.get(r.root_origin, r.root_origin)} ({r.root_origin}), {city_of.get(r.root_origin, '')}: cascade of {int(r.chain_len)} flights "
            f"on aircraft {r.Tail_Number}, started by a flight {int(round(r.root_dep_delay))} minutes late at {int(r.root_local_hour):02d}:00 local in {int(r.Year)}")
cs["description"] = [describe(r) for r in cs.itertuples()]
cs["root_sched_dep_utc"] = pd.to_datetime(cs.root_sched_dep_utc)
cascade_docs = cs.astype({"chain_len": int, "Year": int, "root_local_hour": int, "chain_id": int}).to_dict("records")

collections = {"airports": airports, "summary_airport_hour": summary_docs, "cascade_sample": cascade_docs,
               **{k: v for k, v in small_docs.items()}}
est = {k: sum(len(bson.encode({kk: vv for kk, vv in d.items() if kk != "_id"})) for d in v) / 1e6 for k, v in collections.items()}
for k, v in collections.items(): print(f"{k:22s} {len(v):>8,} docs   ~{est[k]:6.1f} MB (data only)")
total_est = sum(est.values()); print(f"\nestimated data size ~{total_est:.1f} MB (indexes add roughly 30-50%); limit set to {MAX_MB} MB")
assert total_est * 1.5 <= MAX_MB, "too big for the free cluster: lower SAMPLE_N"
print(cascade_docs[0]["description"])

airports                    403 docs   ~   0.1 MB (data only)
summary_airport_hour      5,652 docs   ~   0.6 MB (data only)
cascade_sample           50,000 docs   ~  18.4 MB (data only)
by_year                      17 docs   ~   0.0 MB (data only)
by_hour                      24 docs   ~   0.0 MB (data only)
weather_split                 3 docs   ~   0.0 MB (data only)
delay_classes                 3 docs   ~   0.0 MB (data only)
bts_cause                     5 docs   ~   0.0 MB (data only)

estimated data size ~19.1 MB (indexes add roughly 30-50%); limit set to 120 MB
San Francisco International Airport (SFO), San Francisco: cascade of 2 flights on aircraft N639VA, started by a flight 86 minutes late at 08:00 local in 2014


## 2 · Load into Atlas

In [4]:
if DRY_RUN:
    print("dry run: skipping the load")
else:
    client = MongoClient(URI, serverSelectionTimeoutMS=15000)
    client.admin.command("ping"); db = client[DBNAME]
    print("connected; server", client.server_info()["version"])
    for name, docs in collections.items():
        db.drop_collection(name)                                  # re-runnable: start clean
        col = db[name]
        for i in range(0, len(docs), 5000):
            col.insert_many(docs[i:i+5000], ordered=False)
        print(f"loaded {name:22s} {col.count_documents({}):>8,} documents")
    db.summary_airport_hour.create_index([("airport", 1), ("local_hour", 1)])
    db.cascade_sample.create_index([("root_origin", 1), ("Year", 1)])
    db.airports.create_index([("location", "2dsphere")])
    st = db.command("dbStats"); print(f"\ndatabase size: data {st['dataSize']/1e6:.1f} MB, storage {st['storageSize']/1e6:.1f} MB, indexes {st['indexSize']/1e6:.1f} MB")

connected; server 8.0.39
loaded airports                    403 documents
loaded summary_airport_hour      5,652 documents
loaded cascade_sample           50,000 documents
loaded by_year                      17 documents
loaded by_hour                      24 documents
loaded weather_split                 3 documents
loaded delay_classes                 3 documents
loaded bts_cause                     5 documents

database size: data 20.1 MB, storage 0.8 MB, indexes 0.9 MB


## 3 · Atlas Search indexes
One full-text index on airport name and city, one on the cascade descriptions. The driver creates them; if the cluster refuses, create the same two indexes in the Atlas UI (**Search & Vector Search → Create Search Index**, JSON editor, index name `default`).

In [5]:
import time
SEARCH_DEFS = {
    "airports":       {"mappings": {"dynamic": False, "fields": {"name": {"type": "string"}, "city": {"type": "string"}, "iata": {"type": "string"}}}},
    "cascade_sample": {"mappings": {"dynamic": False, "fields": {"description": {"type": "string"}, "root_origin": {"type": "string"}}}},
}
if DRY_RUN:
    print("dry run: would create", list(SEARCH_DEFS))
else:
    for cname, definition in SEARCH_DEFS.items():
        col = db[cname]
        if not any(ix["name"] == "default" for ix in col.list_search_indexes()):
            col.create_search_index(model=SearchIndexModel(definition=definition, name="default"))
    for cname in SEARCH_DEFS:
        for _ in range(60):                                       # wait up to ~5 minutes
            ready = [ix for ix in db[cname].list_search_indexes() if ix["name"] == "default" and ix.get("queryable")]
            if ready: break
            time.sleep(5)
        print(f"search index on {cname}:", "ready" if ready else "still building (check the Atlas UI)")

search index on airports: ready
search index on cascade_sample: ready


## 4 · Checks: counts, totals, and a first search

In [6]:
if DRY_RUN:
    print("dry run: nothing to check")
else:
    # (1) document counts match what we built
    for name, docs in collections.items():
        assert db[name].count_documents({}) == len(docs), name
    # (2) the hour summary adds up to the cascade total from notebook 03 (3,849,908 cascades, all years)
    total = next(db.summary_airport_hour.aggregate([{"$group": {"_id": None, "n": {"$sum": "$cascades"}}}]))["n"]
    total_from_airports = next(db.airports.aggregate([{"$group": {"_id": None, "n": {"$sum": "$cascades"}}}]))["n"]
    print(f"cascades in summary_airport_hour: {total:,}   in airports: {total_from_airports:,}")
    assert total == total_from_airports
    # (3) Atlas Search: find airports by name, cascades by free text
    print("\nsearch 'dallas' in airports:")
    for d in db.airports.aggregate([{"$search": {"index": "default", "text": {"query": "dallas", "path": ["name", "city"]}}},
                                    {"$limit": 5}, {"$project": {"_id": 0, "iata": 1, "name": 1, "cascades": 1}}]): print("  ", d)
    print("\nsearch 'Chicago aircraft 2019' in cascade_sample:")
    for d in db.cascade_sample.aggregate([{"$search": {"index": "default", "text": {"query": "Chicago 2019", "path": "description"}}},
                                          {"$limit": 3}, {"$project": {"_id": 0, "description": 1}}]): print("  ", d["description"])

cascades in summary_airport_hour: 3,849,908   in airports: 3,849,908

search 'dallas' in airports:
   {'iata': 'DAL', 'name': 'Dallas Love Field', 'cascades': 57791}
   {'iata': 'DFW', 'name': 'Dallas Fort Worth International Airport', 'cascades': 272540}

search 'Chicago aircraft 2019' in cascade_sample:
   Chicago O'Hare International Airport (ORD), Chicago: cascade of 2 flights on aircraft N648RW, started by a flight 57 minutes late at 07:00 local in 2019
   Chicago O'Hare International Airport (ORD), Chicago: cascade of 3 flights on aircraft N657NK, started by a flight 76 minutes late at 11:00 local in 2019
   Chicago O'Hare International Airport (ORD), Chicago: cascade of 2 flights on aircraft N751SK, started by a flight 17 minutes late at 08:00 local in 2019
